In [ ]:
#### Analysis win rate ####
import os
import pandas as pd

base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all"
# base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3"

pickscore_df = pd.read_csv(os.path.join(base_dir, "pickscore/pickscore_score.csv"), dtype={"uid": str})
imagereward_df = pd.read_csv(os.path.join(base_dir, "imagereward/imagereward_score.csv"), dtype={"uid": str})
hpsv3_df = pd.read_csv(os.path.join(base_dir, "hpsv3/hpsv3_score.csv"), dtype={"uid": str})
deqa_df = pd.read_csv(os.path.join(base_dir, "deqa/deqa_score.csv"), dtype={"uid": str})
maagiqa_df = pd.read_csv(os.path.join(base_dir, "ma-agiqa/ma-agiqa_score.csv"), dtype={"uid": str})
aesthetic_df = pd.read_csv(os.path.join(base_dir, "aesthetic/aesthetic_score.csv"), dtype={"uid": str})
anime_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/qwen3_anime/qwen3_anime.csv", dtype={"uid": str})
valid_real_image_uids = anime_df[anime_df['anime'] == 'no']['uid'].tolist()

pickscore_filtered = pickscore_df[pickscore_df['uid'].isin(valid_real_image_uids)]
pickscore_gap_threshold = 0.02
valid_pickscore_uids = pickscore_filtered[pickscore_filtered['real_image_score'] - pickscore_filtered['fake_image_score'] > pickscore_gap_threshold]['uid'].tolist()

valid_uids = valid_pickscore_uids

pickscore_df = pickscore_df[pickscore_df['uid'].isin(valid_uids)]
imagereward_df = imagereward_df[imagereward_df['uid'].isin(valid_uids)]
hpsv3_df = hpsv3_df[hpsv3_df['uid'].isin(valid_uids)]
deqa_df = deqa_df[deqa_df['uid'].isin(valid_uids)]
maagiqa_df = maagiqa_df[maagiqa_df['uid'].isin(valid_uids)]

# 计算每个reward model中满足 real_image_score > fake_image_score 的比例
# 注意：相同分数按0.5计算（平局）
reward_models = {
    'pickscore': pickscore_df,
    'imagereward': imagereward_df,
    'hpsv3': hpsv3_df,
    'deqa': deqa_df,
    'maagiqa': maagiqa_df,
    'aesthetic': aesthetic_df,
}

print(base_dir)
print("=" * 80)
print("Analysis: Percentage of UIDs where real_image_score > fake_image_score")
print("Note: Ties (real == fake) count as 0.5")
print("=" * 80)
print()

results = {}
for model_name, df in reward_models.items():
    total_count = len(df)
    win_count = len(df[df['real_image_score'] > df['fake_image_score']])
    tie_count = len(df[df['real_image_score'] == df['fake_image_score']])
    lose_count = len(df[df['real_image_score'] < df['fake_image_score']])
    score_diff_mean = (df['real_image_score'] - df['fake_image_score']).mean() if total_count > 0 else 0
    
    win_rate = ((win_count + tie_count * 0.5) / total_count * 100) if total_count > 0 else 0
    
    results[model_name] = {
        'total': total_count,
        'win': win_count,
        'tie': tie_count,
        'lose': lose_count,
        'win_rate': win_rate,
        'diff_mean': score_diff_mean
    }
    
    print(f"{model_name:15s}: Win={win_count:6d}, Tie={tie_count:6d}, Lose={lose_count:6d}")
    print(f"{'':15s}  Win Rate = {win_rate:6.2f}% ({(win_count + tie_count * 0.5):.1f} / {total_count})")
    print(f"{'':15s}  Mean(real - fake) = {score_diff_mean:.4f}")
    print()

print("=" * 80)
print("Summary Table")
print("=" * 80)
summary_df = pd.DataFrame(results).T
summary_df.columns = ['Total UIDs', 'Win', 'Tie', 'Lose', 'Win Rate (%)', 'Mean(real - fake)']
print(summary_df.to_string())